In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gamma, ncx2
from scipy.integrate import solve_ivp
from pathlib import Path

figures_dir = Path("figures")
figures_dir.mkdir(exist_ok=True)


In [ ]:
# Base parameters

base_params = {
    "J": 64,                        # number of classes
    "binning": "equal_probability",  # or "tail_refined"
    "Nstar": 1e6,                    # equilibrium population size
    "nubar": 0.9,                    # mean transmission potential
    "k": 1,                          # gamma dispersion
    "delta": 0.2,                    # transmission autocorrelation
    "kernel": "stationary",          # transmission inheritance model
    "sigma": 1 / 3,                  # 1 / latent period
    "gamma": 1 / 4,                  # 1 / infectious period
    "omega": 1 / 365,                # 1 / immune period
    "mu": 1 / (75 * 365),            # 1 / life expectancy
    "season_amp": 0.37,              # transmission varies from +/− amp baseline
    "season_peak": 123,              # day of year for peak transmission
    "import_rate": 1,                # imported infections per day
    "Sinit": 0.6,                    # initial susceptible proportion
    "Iinit": 1,                      # initial infected population
    "tmax": 10 * 365,                # simulation duration in days
    "nt": 10 * 365 + 1,              # number of output times
}

In [ ]:
# Parameter dictionary

def build_params(base):
    J = base["J"]
    nubar = base["nubar"]
    k = base["k"]
    delta = base["delta"]
    kernel = base["kernel"]

    binning = base.get("binning", "equal_probability")
    if binning == "equal_probability":
        quantiles = np.linspace(0, 1, J + 1)
    elif binning == "tail_refined":
        quantiles = np.concatenate((
            np.linspace(0, 0.9, J // 2 + 1),
            np.linspace(0.9, 0.99, J // 4 + 1)[1:],
            np.linspace(0.99, 0.999, J // 8 + 1)[1:],
            (1 - np.geomspace(1e-3, 1e-8, J // 8))[1:],
            [1.0],
        ))
    else:
        raise KeyError(binning)
    b = gamma.ppf(quantiles, k, scale=nubar / k)
    pi = np.diff(gamma.cdf(b, k, scale=nubar / k))
    nu = nubar * np.diff(gamma.cdf(b, k + 1, scale=nubar / k)) / pi
    h = (1 - delta) * nubar + delta * nu

    if kernel == "gamma":
        Q = np.diff(gamma.cdf(b[:, None], k, scale=h[None, :] / k), axis=0)
    elif kernel == "stationary":
        scale = nubar * (1 - delta) / (2 * k)
        nc = 2 * k * delta * nu / (nubar * (1 - delta))
        Q = np.diff(ncx2.cdf(b[:, None], 2 * k, nc[None, :], scale=scale), axis=0)
    else:
        raise KeyError(kernel)

    K = Q * nu[None, :]
    Nnstar = base["Nstar"] * pi
    Iinit = base["Iinit"] * pi
    Sinit = base["Sinit"] * Nnstar
    y0 = np.concatenate((Sinit, np.zeros(J), Iinit, Nnstar - Sinit - Iinit))
    season_amp = base["season_amp"]
    season_peak = base["season_peak"]
    c = lambda t: 1 + season_amp * np.cos(2 * np.pi * (t - season_peak) / 365)

    return base | {
        "binning": binning,
        "quantiles": quantiles,
        "b": b,
        "pi": pi,
        "nu": nu,
        "h": h,
        "Q": Q,
        "K": K,
        "Nnstar": Nnstar,
        "Lambda": base["mu"] * Nnstar,
        "R0": max(abs(np.linalg.eigvals(K))),
        "c": c,
        "y0": y0,
        "t_eval": np.linspace(0, base["tmax"], base["nt"]),
    }

params = build_params(base_params)

In [ ]:
# SEIRS derivatives for state order S, E, I, R

def seirs(t, y, params):
    J = params["J"]
    sigma = params["sigma"]
    gamma = params["gamma"]
    omega = params["omega"]
    mu = params["mu"]
    Nnstar = params["Nnstar"]
    Lambda = params["Lambda"]
    K = params["K"]
    pi = params["pi"]
    import_rate = params["import_rate"]
    c = params["c"](t)

    S, E, I, R = y.reshape(4, J)

    beta = c * (sigma + mu) * (gamma + mu) / sigma * K
    j = S / Nnstar * (beta @ I)
    imports = import_rate * pi

    dS = Lambda - j - imports + omega * R - mu * S
    dE = j - (sigma + mu) * E
    dI = sigma * E + imports - (gamma + mu) * I
    dR = gamma * I - (omega + mu) * R

    return np.concatenate((dS, dE, dI, dR))

In [ ]:
# How we solve the SEIRS model

params = build_params(base_params)

solution = solve_ivp(
    seirs,
    (0, params["tmax"]),
    params["y0"],
    t_eval=params["t_eval"],
    args=(params,),
)

S, E, I, R = solution.y.reshape(4, params["J"], -1)

In [ ]:
# Test SEIRS versus the analytical threshold at three dispersion levels.
from matplotlib.colors import LinearSegmentedColormap

nubar_grid = np.linspace(0.5, 1.5, 101)
delta_grid = np.linspace(0, 0.3, 61)
k_values = [10, 1, 0.1]
heatmap_params = base_params | {"kernel": "stationary", "binning": "tail_refined"}
R0_surfaces = {}

for k in k_values:
    R0_grid = np.empty((len(delta_grid), len(nubar_grid)))
    for i, delta in enumerate(delta_grid):
        for j, nubar in enumerate(nubar_grid):
            p = build_params(heatmap_params | {"k": k, "nubar": nubar, "delta": delta})
            R0_grid[i, j] = p["R0"]
    R0_surfaces[k] = R0_grid

# White near one, softer transitions, and variation within blue and red regions.
color_min, color_max = 0.5, 1.5
color_anchors = [
    (0.5, "#053061"), (0.85, "#4393c3"), (0.9, "#92c5de"),
    (0.99, "white"), (1.01, "white"),
    (1.1, "#f4a582"), (1.15, "#d6604d"), (1.5, "#67001f"),
]
threshold_cmap = LinearSegmentedColormap.from_list(
    "blue_white_red",
    [((value - color_min) / (color_max - color_min), color)
     for value, color in color_anchors],
    N=6001,
)
threshold_fig, threshold_axes = plt.subplots(
    1, 3, figsize=(10, 3), sharex=True, sharey=True, constrained_layout=True,
)
nubar_line = np.linspace(0.5, 1, 501)
for ax, k in zip(threshold_axes, k_values):
    heatmap = ax.pcolormesh(
        nubar_grid, delta_grid, R0_surfaces[k], shading="nearest", cmap=threshold_cmap,
        vmin=color_min, vmax=color_max, rasterized=True,
    )
    delta_star = (1 - nubar_line) / (1 - nubar_line + 1 / k)
    ax.plot(nubar_line, delta_star, color="black", lw=1.8)
    ax.set(xlim=(0.5, 1.5), ylim=(0, 0.3), title=fr"$k={k:g}$")
threshold_axes[0].set_ylabel(r"Transmission correlation $\delta$")
threshold_fig.supxlabel(r"Mean transmission potential $\bar\nu$")
threshold_fig.colorbar(heatmap, ax=threshold_axes, label=r"$R_0=\rho(K)$", extend="both")
threshold_fig.savefig(figures_dir / "seirs_threshold_surfaces.pdf")
plt.show()


In [ ]:
def plot_seirs(params1, params2):
    tmax = 15 * 365
    t_eval = np.arange(tmax + 1)
    plot_start = 10 * 365

    ts_fig = plt.figure(figsize=(10, 5), constrained_layout=True)
    grid = ts_fig.add_gridspec(2, 2)
    top = ts_fig.add_subplot(grid[0, 0])
    bottom = ts_fig.add_subplot(grid[1, 0], sharex=top)
    phase = ts_fig.add_subplot(grid[:, 1])
    ts_axes = np.array([top, bottom, phase])
    top.tick_params(labelbottom=False)

    # Keep each parameter set's color while drawing delta=0.2 last.
    runs = [(params1, "#3B6F8E"), (params2, "#C45A43")]
    runs.sort(key=lambda run: bool(np.isclose(run[0]["delta"], 0.2)))
    for params, color in runs:
        solution = solve_ivp(
            seirs,
            (0, tmax),
            params["y0"],
            t_eval=t_eval,
            args=(params,),
        )
        
        sus, _, inf, _ = solution.y.reshape(4, params["J"], -1)
        years = solution.t / 365
        N = params["Nstar"]
        keep = solution.t >= plot_start
        phase.plot(
            sus.sum(axis=0)[keep] / N, inf.sum(axis=0)[keep],
            color=color, lw=1.5, alpha=1,
            label=rf"$\delta={params['delta']:g}$",
        )

        top.plot(years, 1000 * inf.sum(axis=0) / N, color=color, lw=1.5,
                 label=rf"$\delta={params['delta']:g}$")

        # Include the fraction of any class that crosses the 99th percentile.
        upper_quantiles = np.cumsum(params["pi"])
        top_one_percent = np.clip((upper_quantiles - 0.99) / params["pi"], 0, 1)
        susceptible_top = top_one_percent @ sus[:, keep]
        bottom.plot(
            years[keep], susceptible_top / N, color=color, lw=1.5,
            label=rf"$\delta={params['delta']:g}$",
        )

    top.set_ylabel(r"$I/N \times 1000$")
    bottom.set(xlabel="Time (years)", ylabel=r"$S_{>99}/N$")

    # Fit both infectious series in the displayed window, with room for labels.
    peak = max(line.get_ydata()[(line.get_xdata() >= plot_start / 365)
                              & (line.get_xdata() <= tmax / 365)].max()
               for line in top.lines)
    top.set_ylim(0, max(1.35 * peak, 0.01))
    top.set_xlim(plot_start / 365, tmax / 365)

    # Fit both post-burn-in susceptible fractions with padding.
    ymin = min(line.get_ydata().min() for line in bottom.lines)
    ymax = max(line.get_ydata().max() for line in bottom.lines)
    padding = max(0.15 * (ymax - ymin), 0.01 * ymax, 1e-6)
    bottom.set_ylim(max(0, ymin - padding), ymax + 2 * padding)
    phase.set(xlabel=r"$S/N$", ylabel=r"$I$",
              xscale="linear", yscale="log")
    phase.set_xticks([0.8, 0.85, 0.9, 0.95], labels=["0.8", "0.85", "0.9", "0.95"])
    # Reserve space above the trajectories in log coordinates for panel c.
    lo, hi = np.log10(phase.get_ylim())
    phase.set_ylim(10 ** lo, 10 ** (hi + 0.25 * (hi - lo)))
    phase.legend(frameon=False, fontsize=8 * 1.25)

    # Shared seasonal range: the two runs have matched R0 and forcing.
    r0 = params1["R0"]
    amp = abs(params1["season_amp"])
    rmin, rmax = (1 - amp) * r0, (1 + amp) * r0
    span = max(rmax - rmin, 0.1)
    r_ax = top.inset_axes([0.18, 1.06, 0.76, 0.36])
    r_ax.set(xlim=(rmin - 0.15 * span, rmax + 0.15 * span), ylim=(0, 1.2))
    r_ax.axis("off")
    r_ax.hlines(0.5, rmin, rmax, color="#555555", lw=1.2)
    r_ax.vlines([rmin, rmax], 0.36, 0.64, color="#555555", lw=1.2)
    if rmin <= 1 <= rmax:
        r_ax.vlines(1, 0.32, 0.68, color="#aaaaaa", lw=1, linestyles="dashed")
        r_ax.text(1, 0.72, "1", ha="center", va="bottom", fontsize=7 * 1.25, color="#777777")
    r_ax.plot(r0, 0.5, "o", color="#333333", ms=4)
    for x, name in [(rmin, "Seasonal min"), (r0, r"$R_0$"), (rmax, "Seasonal max")]:
        r_ax.text(x, 1.08, name, ha="center", va="center", fontsize=8 * 1.25)
        r_ax.text(x, 0.04, f"{x:.2f}", ha="center", va="center", fontsize=8 * 1.25)
    for letter, ax in zip("abc", ts_axes):
        ax.text(0.03, 0.95, letter, transform=ax.transAxes,
                ha="left", va="top", fontweight="bold", fontsize=11,
                bbox=dict(facecolor="white", edgecolor="none", pad=1))
        ax.spines[["top", "right"]].set_visible(False)
    # Enlarge all figure text, including inset annotations and tick labels.
    from matplotlib.text import Text

    ts_fig.canvas.draw()
    for text in ts_fig.findobj(match=Text):
        text.set_fontsize(1.25 * text.get_fontsize())

    plt.show()
    return ts_fig, ts_axes


In [ ]:
params1 = build_params(base_params | {"nubar": 0.85})
params2 = build_params(base_params | {"delta": 0, "nubar": params1["R0"]})

ts_fig, ts_axes = plot_seirs(params1, params2)


In [ ]:
# Save the time-series figure in the repository's figures directory.
output_path = figures_dir / "seirs_timeseries.pdf"
ts_fig.savefig(output_path, 
               format="pdf", 
               bbox_inches="tight")